In [59]:
import re, math,random,sys,os
import pandas as pd
import numpy as np
from stdnum import figi as stdnum_figi

In [52]:
from openfigi import api_call

## Figi

In [21]:
# create a functions that validates a given string is a valid FIGI
# A valid FIGI is a 12-character alphanumeric string that with a check digit.
# Validation tests:
# 1. Length check: The FIGI must be exactly 12 characters long.
# 2. Alphanumeric check: The FIGI must consist of only letters and digits.
# 3. Check digit validation: The last character of the FIGI is a check digit, which is calculated using the preceding 11 characters. The check digit is calculated using the Luhn algorithm, which is a common method for validating identification numbers.
# 4. The function should return True if the FIGI is valid and False otherwise.
# 5. The function should also handle edge cases, such as empty strings, strings with special characters, and strings that are too short or too long.
# 6. The function should be case-insensitive, meaning that it should treat uppercase and lowercase letters as equivalent.
# 7. The function should be efficient and able to handle large volumes of FIGI validations without significant performance degradation.
# Example usage:
# print(is_valid_figi("BBG000BLNNH6"))  # True
# print(is_valid_figi("BBG000BLNNH7"))  # False
# print (is_valid_figi("BBG000BLNQ16"))   # False

def is_valid_figi(figi: str) -> bool:
    if len(figi) != 12:
        return False
    if not figi.isalnum():
        return False
    
    # check digit validation using Luhn algorithm
    def calculate_check_digit(figi_without_check_digit: str) -> str:
        digits = [int(char) if char.isdigit() else ord(char.upper()) - ord('A') + 10 for char in figi_without_check_digit] #convert characters to digits
        digits = [digit * (1,2)[i % 2] for i, digit in enumerate(digits)] # double every second digit from the right
        total = sum([int(c) for c in ''.join(map(str,digits))]) # sum all the digits individually, numbers greater than 9 are split into their individual digits
        next_total = total + (10 - total % 10) # calculate the next multiple of 10
        check_digit = next_total - total # calculate the check digit, i.e. how much to get to next multiple of 10
        return str(check_digit)
    
    # Calculate the check digit
    check_digit = figi[-1]
    calculated_check_digit = calculate_check_digit(figi[:-1])
    return check_digit == calculated_check_digit

In [57]:
# testing the function with some examples
figis = [
    'BBG000BLNNH6',
    'BBG000BLNNH7',
    'BBG000BLNQ16'
]
print("testing my function")
for f in figis:
    print(is_valid_figi(f))
print("testing stdnum function")
for f in figis:
    print(stdnum_figi.is_valid(f))
print("testing api_call function")
for f in figis:
    mapping_request = [
      {"idType":"ID_BB_GLOBAL","idValue":f},
    ]
    print(api_call(path="/v3/mapping", data=mapping_request))

testing my function
True
False
True
testing stdnum function
True
False
True
testing api_call function
[{'data': [{'figi': 'BBG000BLNNH6', 'name': 'INTL BUSINESS MACHINES CORP', 'ticker': 'IBM', 'exchCode': 'US', 'compositeFIGI': 'BBG000BLNNH6', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG001S5S399', 'securityType2': 'Common Stock', 'securityDescription': 'IBM'}]}]
[{'warning': 'No identifier found.'}]
[{'data': [{'figi': 'BBG000BLNQ16', 'name': 'INTL BUSINESS MACHINES CORP', 'ticker': 'IBM', 'exchCode': 'UN', 'compositeFIGI': 'BBG000BLNNH6', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG001S5S399', 'securityType2': 'Common Stock', 'securityDescription': 'IBM'}]}]


## Max/Min of Subsets of Arrays

In [24]:
# calculating max of sum of contigous subarrays of size k, i.e. sliding window of size k
nums = [9,4,1,7,5,8,23,56,98,12,7,8,9,9]
k = 3
for i in range(len(nums)-k+1):
    lst = nums[i:i+k]
    print(f'subarray: {lst}, first: {lst[0]}, last: {lst[-1]}, difference: {max(lst) - min(lst)}')

subarray: [9, 4, 1], first: 9, last: 1, difference: 8
subarray: [4, 1, 7], first: 4, last: 7, difference: 6
subarray: [1, 7, 5], first: 1, last: 5, difference: 6
subarray: [7, 5, 8], first: 7, last: 8, difference: 3
subarray: [5, 8, 23], first: 5, last: 23, difference: 18
subarray: [8, 23, 56], first: 8, last: 56, difference: 48
subarray: [23, 56, 98], first: 23, last: 98, difference: 75
subarray: [56, 98, 12], first: 56, last: 12, difference: 86
subarray: [98, 12, 7], first: 98, last: 7, difference: 91
subarray: [12, 7, 8], first: 12, last: 8, difference: 5
subarray: [7, 8, 9], first: 7, last: 9, difference: 2
subarray: [8, 9, 9], first: 8, last: 9, difference: 1


## Pandas

In [33]:
student_data = [
    [1,15],
    [2,11],
    [3,11],
    [4,20]
]
student_data = pd.DataFrame(student_data,columns=['id','score'])
student_data

,id,score
0,1,15
1,2,11
2,3,11
3,4,20


In [34]:
# dimensions of the dataframe
list(student_data.shape)

[4, 2]

In [48]:
# first two rows of the dataframe
student_data.iloc[0:2]

,id,score
0,1,15
1,2,11


In [47]:
# add a new column 'bonus' which is double the score
student_data.assign(bonus=student_data['score']*2)

,id,score,bonus
0,1,15,30
1,2,11,22
2,3,11,22
3,4,20,40


In [38]:
# drop duplicate rows based on the 'score' column, keeping only unique scores
student_data.drop_duplicates(subset=['score'],keep=False)

,id,score
0,1,15
3,4,20


In [39]:
# drop duplicate rows based on the 'score' column, keeping the first occurrence of each unique score
student_data.drop_duplicates(subset=['score'],keep='first')

,id,score
0,1,15
1,2,11
3,4,20


In [40]:
# rename the 'score' column to 'grade'
student_data.rename(columns={'score':'grade'},inplace=False)

,id,grade
0,1,15
1,2,11
2,3,11
3,4,20


In [ ]:
# concatenate, add rows, the original dataframe with a new dataframe containing additional student data, ignoring the index to create a continuous index in the resulting dataframe
pd.concat([student_data,pd.DataFrame({'id':[5,6],'score':[10,12]})],ignore_index=True)

,id,score
0,1,15
1,2,11
2,3,11
3,4,20
4,5,10
5,6,12


In [42]:
weather = pd.DataFrame({
    'city': ['JacksonVile']*5 + ['ElPaso']*5,
    'month': ['January', 'February', 'March', 'April', 'May','January', 'February', 'March', 'April', 'May'],
    'temperature': [70, 75, 80, 85, 90, 65, 70, 75, 80, 85]
})
weather

,city,month,temperature
0,JacksonVile,January,70
1,JacksonVile,February,75
2,JacksonVile,March,80
3,JacksonVile,April,85
4,JacksonVile,May,90
5,ElPaso,January,65
6,ElPaso,February,70
7,ElPaso,March,75
8,ElPaso,April,80
9,ElPaso,May,85


In [43]:
# pivot the dataframe to have cities as columns, months as index, and temperature as values
weather_pivot = weather.pivot(index='month',columns='city',values='temperature')
weather_pivot

city,ElPaso,JacksonVile
month,,
April,80,85
February,70,75
January,65,70
March,75,80
May,85,90


In [49]:
products = pd.DataFrame({
    'product': ['umbrella','sleeping_bag'],
    'quarter_1': [417,800],
    'quarter_2': [224,936],
    'quarter_3': [379,93],
    'quarter_4': [611,875]
})
products

,product,quarter_1,quarter_2,quarter_3,quarter_4
0,umbrella,417,224,379,611
1,sleeping_bag,800,936,93,875


In [50]:
# melt the dataframe to have 'product' as id_vars, 'quarter' as var_name, and 'sales' as value_name, then sort the resulting dataframe by 'sales' in descending order
products.melt(id_vars='product',var_name='quarter',value_name='sales').sort_values(by='sales',ascending=False)

,product,quarter,sales
3,sleeping_bag,quarter_2,936
7,sleeping_bag,quarter_4,875
1,sleeping_bag,quarter_1,800
6,umbrella,quarter_4,611
0,umbrella,quarter_1,417
4,umbrella,quarter_3,379
2,umbrella,quarter_2,224
5,sleeping_bag,quarter_3,93


## Scratch

In [139]:
pattern = '(?<=[a-zA-Z0-9])[\!\@\#\$\%\&\s]+(?=[a-zA-Z0-9])'
text = ''.join(['T', 'h', 'i', 's', '$', '#', 'i', 's', '%', ' ', 'M', 'a', 't', 'r', 'i', 'x', '#', ' ', ' ', '%', '!'])
text

'This$#is% Matrix#  %!'

In [140]:
re.findall(pattern,text)

['$#', '% ']

In [141]:
result = re.sub(pattern, ' ',text)
result

'This is Matrix#  %!'

In [142]:
test0_result = 'This is Matrix#  %!'
test0_result

'This is Matrix#  %!'

In [134]:
result == test0_result

True

In [ ]:
t = '9feet 10'
re.findall(r'\d+(?=[a-z])',t)

['9']

In [117]:
t = 'this#$is%matrix'
re.findall('(?![a-z])[\!\@\#\$\%\&]+(?=[a-z])',t)

['#$', '%']

In [125]:
'This is Matrix#  %!'

'This is Matrix#  %!'

In [126]:
'This is Matrix#  %!'

'This is Matrix#  %!'